# Tool Menu Overload Regression Eval

This notebook demonstrates and quantifies how an agent model’s tool-use quality degrades as the number of available tool options increases—even when the task does not require the newly-added tools.

**Primary question:** Is going from **14 → 17 tools** likely to be problematic for existing use cases that don’t need those 3 new tools?

**What “problematic” means in this notebook:**

- Lower task success rate on a fixed suite of tasks
- More wrong-tool selections
- More incorrect sequencing (calls out of order)
- More invalid arguments / schema errors
- More unnecessary tool calls (cost/latency blow-ups)
- More “tool confusion” between overlapping tools


## 1) Planning scenario (eval task)

### Scenario: “Customer Visit Trip + Calendar Coordination”

**User request template (varies across tasks):**

> “Plan my 2-day trip to visit Customer X in Seattle next week. I need a flight that arrives before 3pm local time on Day 1, a hotel near the customer site, schedule two meetings with their team, and email my manager the final itinerary.”

### Core tools required (the “base menu”)

1. `get_user_profile()` → home airport, preferences  
2. `get_calendar(date_range)` → availability windows  
3. `search_flights(origin, destination, depart_date, return_date, constraints)`  
4. `book_flight(flight_id, traveler_info)`  
5. `search_hotels(city, checkin, checkout, constraints)`  
6. `book_hotel(hotel_id, guest_info)`  
7. `create_calendar_event(title, start, end, attendees, location)`  
8. `send_email(to, subject, body)`

### Ground-truth correct sequence (canonical path)

1. `get_user_profile`
2. `get_calendar`
3. `search_flights`
4. `book_flight`
5. `search_hotels`
6. `book_hotel`
7. `create_calendar_event` (meeting 1)
8. `create_calendar_event` (meeting 2)
9. `send_email` (itinerary summary)


## 2) Hypotheses

- **H1: Tool count hurts performance on unchanged tasks.** Success drops as the menu grows (e.g., 8 → 14 → 17 → 25) even though tasks require only the original 8 tools.
- **H2: Similar/overlapping tools cause disproportionate confusion.** Overlapping tools degrade performance more than irrelevant tools.
- **H3: Context bloat matters.** Longer tool descriptions cause failures even when tool count stays at 8.


## 3) Experimental design

### Conditions

- **A: Minimal tools (baseline)** — 8 required tools
- **B: +irrelevant tools** — 8 + 6 (14 total), 8 + 9 (17 total)
- **C: +overlapping tools** — 8 + 6 (14 total), 8 + 9 (17 total)
- **D: Token-length control** — still 8 tools, but padded descriptions to match token count of 17-tool condition
- **E: Tool ordering randomization** — shuffle tool order each run

### Example overlapping tools

- `search_flight_deals` (overlaps with `search_flights`)
- `reserve_flight` (overlaps with `book_flight`)
- `find_hotels` (overlaps with `search_hotels`)
- `draft_email` (overlaps with `send_email`)
- `create_meeting` (overlaps with `create_calendar_event`)


## 4) Task suite design

- **N = 100–300 tasks** (synthetic but realistic)
- Each task varies:
  - dates
  - time constraints (arrive before 3pm, avoid red-eye, etc.)
  - meeting attendee availability patterns
  - budget constraints
  - “must email manager” vs “must email customer”

### Task format

```json
{
  "task_id": "trip_042",
  "prompt": "Plan my 2-day trip to visit Customer X ...",
  "constraints": {
    "arrive_by": "15:00",
    "budget_max": 900,
    "meeting_count": 2
  },
  "gold": {
    "required_tools": [
      "get_user_profile",
      "get_calendar",
      "search_flights",
      "book_flight",
      "search_hotels",
      "book_hotel",
      "create_calendar_event",
      "send_email"
    ],
    "canonical_sequence": [
      "get_user_profile",
      "get_calendar",
      "search_flights",
      "book_flight",
      "search_hotels",
      "book_hotel",
      "create_calendar_event",
      "create_calendar_event",
      "send_email"
    ]
  }
}
```


## 5) Tool simulator (deterministic world)

- Simulated flight database
- Simulated hotel database
- Simulated calendar with busy blocks
- Simulated email outbox log

Each tool returns structured outputs with IDs. State is updated only by booking tools and event creation. Tools validate inputs and return errors for invalid sequences.


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

@dataclass
class Flight:
    flight_id: str
    origin: str
    destination: str
    depart: str
    arrive: str
    price: int

@dataclass
class Hotel:
    hotel_id: str
    city: str
    name: str
    nightly_rate: int

@dataclass
class WorldState:
    flights: List[Flight] = field(default_factory=list)
    hotels: List[Hotel] = field(default_factory=list)
    booked_flight_id: Optional[str] = None
    booked_hotel_id: Optional[str] = None
    calendar_events: List[Dict] = field(default_factory=list)
    sent_emails: List[Dict] = field(default_factory=list)

    last_flight_search_ids: List[str] = field(default_factory=list)
    last_hotel_search_ids: List[str] = field(default_factory=list)


In [ ]:
# Example deterministic data (expand for full evals)
FLIGHTS = [
    Flight("FL1", "SFO", "SEA", "2025-04-08T09:00", "2025-04-08T11:00", 320),
    Flight("FL2", "SFO", "SEA", "2025-04-08T12:00", "2025-04-08T14:00", 410),
    Flight("FL3", "SFO", "SEA", "2025-04-08T16:00", "2025-04-08T18:00", 280),
]

HOTELS = [
    Hotel("HT1", "SEA", "Pike Place Inn", 190),
    Hotel("HT2", "SEA", "Lake Union Suites", 220),
]


In [ ]:
# Tool implementations (pure functions + state transitions)

def get_user_profile():
    return {"home_airport": "SFO", "preferences": {"seat": "aisle"}}


def get_calendar(date_range: Tuple[str, str]):
    return {"busy": [("2025-04-08T15:30", "2025-04-08T16:30")]}


def search_flights(origin: str, destination: str, depart_date: str, return_date: str, constraints: Dict, state: WorldState):
    matches = [f for f in state.flights if f.origin == origin and f.destination == destination]
    state.last_flight_search_ids = [f.flight_id for f in matches]
    return {"results": [f.__dict__ for f in matches]}


def book_flight(flight_id: str, traveler_info: Dict, state: WorldState):
    if flight_id not in state.last_flight_search_ids:
        return {"error": "flight_id not in prior search results"}
    state.booked_flight_id = flight_id
    return {"booked_flight_id": flight_id}


def search_hotels(city: str, checkin: str, checkout: str, constraints: Dict, state: WorldState):
    matches = [h for h in state.hotels if h.city == city]
    state.last_hotel_search_ids = [h.hotel_id for h in matches]
    return {"results": [h.__dict__ for h in matches]}


def book_hotel(hotel_id: str, guest_info: Dict, state: WorldState):
    if hotel_id not in state.last_hotel_search_ids:
        return {"error": "hotel_id not in prior search results"}
    state.booked_hotel_id = hotel_id
    return {"booked_hotel_id": hotel_id}


def create_calendar_event(title: str, start: str, end: str, attendees: List[str], location: str, state: WorldState):
    event = {"title": title, "start": start, "end": end, "attendees": attendees, "location": location}
    state.calendar_events.append(event)
    return {"event_id": f"EV{len(state.calendar_events)}"}


def send_email(to: str, subject: str, body: str, state: WorldState):
    msg = {"to": to, "subject": subject, "body": body}
    state.sent_emails.append(msg)
    return {"status": "sent"}


## 6) Model runner harness

The harness should log for each (task, condition, seed):

- Full tool list presented to the model (names + descriptions)
- Model messages
- Tool calls (name, args, timestamp)
- Tool responses
- Final model answer
- Evaluation results and failure reason

Below is a scaffolding interface (wire to your actual agent runner).


In [ ]:
from typing import Any

@dataclass
class ToolCall:
    name: str
    args: Dict[str, Any]

@dataclass
class EpisodeLog:
    task_id: str
    condition: str
    seed: int
    tool_list: List[str]
    tool_calls: List[ToolCall]
    tool_responses: List[Dict[str, Any]]
    success: bool
    failure_reason: Optional[str]


def run_episode(task: Dict[str, Any], condition: str, seed: int) -> EpisodeLog:
    # TODO: integrate with your agent framework
    # For now, return a placeholder log.
    return EpisodeLog(
        task_id=task["task_id"],
        condition=condition,
        seed=seed,
        tool_list=[],
        tool_calls=[],
        tool_responses=[],
        success=False,
        failure_reason="not_implemented",
    )


## 7) Scoring and metrics

- **Task Success Rate** (binary)
- **Wrong-tool rate**
- **New-tool intrusion rate**
- **Tool confusion rate** (equivalence map)
- **Order validity**
- **Schema validation pass rate**
- **Tool calls per success**


In [ ]:
import pandas as pd


def score_episode(log: EpisodeLog) -> Dict[str, Any]:
    # TODO: implement detailed scoring
    return {
        "task_id": log.task_id,
        "condition": log.condition,
        "seed": log.seed,
        "success": log.success,
        "failure_reason": log.failure_reason,
    }


def summarize_scores(scores: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(scores)
    return df.groupby(["condition"]).agg(success_rate=("success", "mean"))


## 8) Analyses and plots

The notebook should produce:

1. **Success vs tool count** (line chart)
2. **New tool intrusion vs tool count**
3. **Confusion matrix of intended tool vs selected tool**
4. **Error taxonomy bar chart**
5. **Statistical summary with confidence intervals**
6. **Power / sensitivity analysis** to detect 2–5 point drops


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# TODO: fill with real scores
example = pd.DataFrame({
    "tool_count": [8, 14, 17, 25],
    "success_rate": [0.85, 0.82, 0.78, 0.72],
    "condition": ["irrelevant"] * 4,
})

fig, ax = plt.subplots()
for condition, group in example.groupby("condition"):
    ax.plot(group["tool_count"], group["success_rate"], marker="o", label=condition)
ax.set_xlabel("Tool count")
ax.set_ylabel("Success rate")
ax.set_title("Success vs Tool Count")
ax.legend()
plt.show()


## 9) Decision gate

Add a final cell that produces a CI-style “ship/no-ship” view:

**Example policy:**

- Block shipping if:
  - success drops by >2 points on the unchanged task suite, OR
  - new-tool intrusion rate >1%, OR
  - wrong-tool rate increases by >X relative


In [ ]:
# Example decision gate

def decision_gate(success_drop: float, new_tool_intrusion: float, wrong_tool_increase: float) -> str:
    if success_drop > 0.02 or new_tool_intrusion > 0.01 or wrong_tool_increase > 0.02:
        return "BLOCK"
    return "SHIP"

print(decision_gate(success_drop=0.04, new_tool_intrusion=0.015, wrong_tool_increase=0.01))
